# Lab 4 — Cost & prompt caching (EP08)

From the theory video (§18.8): the cost of one task (Equation 18.16) and **prompt caching** —
the provider keeps the work it already did on **an exactly repeated beginning**, so the rule is *static first, changing last*

**What you'll learn**
1. Work out each call's cost from `usage` — counting cached tokens separately (they're cheaper)
2. A long system prompt (rules + a 120-paper catalog) and eight questions: the current time **at the end** vs **on the first line** — reading `cached_tokens`
3. A task's cost by Equation 18.16: the model cost of every call + the cost of every tool call
4. A token budget per task: warn when close, stop when over

The paper catalog and the tool prices are **simulated data for teaching**. Token counts, `cached_tokens` and the answers come from real runs.

> Source: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §18.8 — CC BY-SA 4.0

> ✏️ **Starter notebook — you write the key parts**
>
> Most of the code is ready. Write only the **4 places marked `✏️ TODO`** (replace `___` or `...` with your code) — each one gives the time in the video where that code is explained (▶ m:ss). Watch the video first, then try it yourself without opening the solution.
>
> Run every cell from top to bottom. A `NameError` or `TypeError` in a cell you have not finished yet is normal.
>
> Stuck? The solution is **`ep08-lab4-cost-and-caching.en.ipynb`** (the same code as in the video).

## 0. Setup

| You need | Details |
|---|---|
| Python 3.10 or later | or Google Colab, which already has it |
| An OpenRouter API key | the same key as Lab 1 (this lab costs about 1 US cent) |
| Packages | `openai`, `python-dotenv` |

How to add the key is in **Lab 1** — which also explains why this EP pins the provider to Makora: **because of this lab**

In [ ]:
%pip install -q "openai>=1.50" python-dotenv

In [ ]:
import os, random, uuid
from datetime import datetime
from openai import OpenAI

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key())
MODEL = "google/gemma-4-26b-a4b-it"
EXTRA = {"reasoning": {"enabled": False}, "provider": {"order": ["makora"], "allow_fallbacks": False}}

## 1. The cost of one call

Makora's prices per million tokens (see the model's page on OpenRouter → Providers):

| | USD / 1M tokens |
|---|---|
| input | 0.09 |
| cached input (cache read) | 0.049 |
| output | 0.28 |

`usage.prompt_tokens_details.cached_tokens` = how many input tokens the provider read from its cache

In [ ]:
# ✏️ TODO 1 (▶ video 0:59): price it: fresh tokens × input price + cached tokens × cache price + output tokens × output price
PRICE = {"in": 0.09, "cached": 0.049, "out": 0.28}   # USD per 1M tokens (Makora)

def call_cost(usage):
    details = usage.prompt_tokens_details
    cached = (details.cached_tokens or 0) if details else 0
    fresh = usage.prompt_tokens - cached
    return ___, cached

## 2. Prompt caching: static part first vs the time on the first line

System prompt = rules + a simulated catalog of 120 papers (about 4,600 tokens), **the same on every call** — like the system prompt + tool definitions in the theory video.
`SNAPSHOT` is a random id for this catalog, so re-running the notebook doesn't hit cache left over from a previous run

In [ ]:
random.seed(7)
SNAPSHOT = uuid.uuid4().hex[:6]
TOPICS = ["forecasting", "anomaly detection", "imputation", "classification", "foundation models"]
CATALOG = "\n".join(
    f"- P{i}: {random.choice(['Transformer', 'MLP', 'CNN', 'State-space'])} model for time-series {random.choice(TOPICS)} "
    f"({2018 + i % 7}). Benchmarks: ETTh1, Weather, Traffic. Main idea: "
    f"{random.choice(['patching', 'sparse attention', 'decomposition', 'frequency mixing', 'pre-training on many datasets'])}."
    for i in range(1, 121))
RULES = ("You are the Cogito Learn research assistant.\n"
         "Rules: cite paper ids like [P12]; answer in under 60 words; say 'not in the catalog' if unsure.\n\n"
         f"Paper catalog (snapshot {SNAPSHOT}):\n" + CATALOG)
QUESTIONS = ["Which papers use patching?", "Name one anomaly detection paper.", "Which papers are from 2024?",
             "Any state-space models?", "Which paper is about imputation?", "Name a CNN paper.",
             "Which papers do frequency mixing?", "Name an MLP paper about forecasting."]

In [ ]:
def ask(system, user):
    reply = client.chat.completions.create(model=MODEL, max_tokens=80, extra_body=EXTRA,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}])
    return reply.usage

def run(name, build):
    total, hits = 0.0, 0
    for q in QUESTIONS:
        system, user = build(q)
        usage = ask(system, user)
        cost, cached = call_cost(usage)
        total += cost; hits += cached > 0
        print(f"  {usage.prompt_tokens:,} in · cached {cached:,} · ${cost:.6f}")
    print(f"{name}: {hits}/{len(QUESTIONS)} calls hit the cache · ${total:.5f}\n")
    return total

In [ ]:
# ✏️ TODO 2 (▶ video 2:14): put the rules (static) first and the current time (always changing) at the end of the user message
def static_first(q):
    return ___, f"{q}\n(Current time: {datetime.now():%H:%M:%S})"

cost_static = run("static first", static_first)

In [ ]:
def time_first(q):
    return f"Current time: {datetime.now():%H:%M:%S}\n" + RULES, q

cost_time = run("time first", time_first)

In [ ]:
print(f"static first ${cost_static:.5f} vs time first ${cost_time:.5f} → saved {1 - cost_static / cost_time:.0%}")

## 3. The cost of the whole task (Equation 18.16)

$$\text{Cost}_{task} = \sum_{i=1}^{T} \left(p_{in} \cdot n_{in,i} + p_{out} \cdot n_{out,i}\right) + \sum_{j=1}^{K} c_j$$

The model cost of every call (T of them) + the cost of every tool call (K of them) · assume that before each answer the agent calls a search API costing USD 0.001 per call (a simulated price)

In [ ]:
# ✏️ TODO 3 (▶ video 3:40): tool cost = number of calls × price per call
TOOL_PRICE = 0.001          # USD per search call (simulated)
llm_cost = cost_static      # the eight model calls, static first
tool_cost = ___
print(f"LLM ${llm_cost:.5f} + tools ${tool_cost:.5f} = ${llm_cost + tool_cost:.5f} per task")

## 4. A token budget per task

Textbook §18.8.2: enforce a token budget per task (and per user) — **alert when close** · stop when over

In [ ]:
# ✏️ TODO 4 (▶ video 4:18): add this call's tokens, and stop with an error when over the limit
class TokenBudget:
    def __init__(self, limit, warn_at=0.8):
        self.limit, self.warn_at, self.used = limit, warn_at, 0

    def charge(self, usage):
        self.used += ___
        if ___:
            raise RuntimeError(f"token budget exceeded: {self.used:,} > {self.limit:,}")
        if self.used > self.warn_at * self.limit:
            print(f"  ⚠️ {self.used:,} of {self.limit:,} tokens used")

In [ ]:
budget = TokenBudget(limit=20_000)
try:
    for q in QUESTIONS:
        budget.charge(ask(*static_first(q)))
        print(f"✓ {q}  ({budget.used:,} tokens so far)")
except RuntimeError as e:
    print("🛑", e)

## Summary

| What we did | What we saw in the lab |
|---|---|
| Cost per call | Cached tokens counted separately: Makora charges about 46% less for them (the textbook says 50–90% — it depends on the provider) |
| Static first | The first call isn't cached · many after it are (not all: the provider runs several machines) · a cached call costs almost half |
| Time on the first line | Not a single cache hit, even though the four thousand-plus tokens after it never changed |
| Equation 18.16 | The tools cost about three times the model — count both parts |
| Token budget | A warning at 80% · a stop when over · caching lowers the price, not the tokens |

**Exercises**
1. **Model routing:** send the short questions to `google/gemma-3-12b-it` (provider `deepinfra/bf16`, Lab 3) instead, and compare cost and answer quality
2. Put the current time in the **middle** of the system prompt (after the rules, before the catalog) — how many tokens get cached, and why?

**Next — Lab 5:** Lab 2's checkpoints again, with LangGraph — crash midway, resume, and look back through every checkpoint